In [12]:
import os
from os import path

from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

# Load environment variables from the project's .env file.
load_dotenv()

# Initialize Gemini through LangChain.
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0.3,
    max_retries=2
)

In [2]:
#%pip install -U langchain-text-splitters langchain-chroma chromadb pypdf

In [5]:
#pip show pypdf

In [8]:
# pip install langchain-community

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [14]:
from pathlib import Path

In [15]:
# --------------------------------------------------
# 3. Load PDF documents
# --------------------------------------------------

pdf_files = [
    "remote_work_policy.pdf",
    "business_travel_and_expenses.pdf"
]

documents = []

for pdf_file in pdf_files:

    if not Path(pdf_file).exists():
        raise FileNotFoundError(f"PDF not found: {pdf_file}")

    loader = PyPDFLoader(pdf_file)

    loaded_pages = loader.load()

    documents.extend(loaded_pages)

print("Total pages loaded:", len(documents))


Total pages loaded: 3


In [ ]:
print(documents)

In [18]:
# --------------------------------------------------
# 4. Split documents into chunks
# --------------------------------------------------

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)
print("Total chunks created:", len(chunks))

Total chunks created: 13


In [19]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [20]:
embeddings = GoogleGenerativeAIEmbeddings(model='models/gemini-embedding-001')

In [22]:
# --------------------------------------------------
# 5. Create vector store
# --------------------------------------------------
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="northstar_rag_demo",
    persist_directory='./mydb.db'
)
print("Vector store created successfully.")

Vector store created successfully.


In [23]:
# --------------------------------------------------
# 6. Create retriever
# --------------------------------------------------
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

In [24]:
# --------------------------------------------------
# 7. Create prompt
# --------------------------------------------------
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are a document question-answering assistant.

        Answer the question using ONLY the supplied context.

        If the answer is not available in the context,
        clearly state that the documents do not contain
        sufficient information.

        Do not invent facts.

        Context:
        {context}
        """
    ),
    ("human", "{question}")
])

In [25]:
# --------------------------------------------------
# 8. Format retrieved documents
# --------------------------------------------------
def format_docs(docs):

    return "\n\n".join(
        f"Source: {doc.metadata.get('source', 'Unknown')}\n"
        f"Page: {doc.metadata.get('page', 'Unknown')}\n"
        f"Content: {doc.page_content}"
        for doc in docs
    )

In [26]:
# --------------------------------------------------
# 9. Build RAG chain
# --------------------------------------------------
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
# --------------------------------------------------
# 10. Ask a question and inspect retrieval
# --------------------------------------------------

question = "How many days must an employee work before requesting remote work?"

retrieved_docs = retriever.invoke(question)

print("\n========== RETRIEVED DOCUMENTS ==========\n")

for index, doc in enumerate(retrieved_docs, start=1):

    print(f"Document {index}")
    print("Source:", doc.metadata.get("source"))
    print("Page:", doc.metadata.get("page"))
    print("Content:", doc.page_content)
    print("-" * 60)

In [28]:
# --------------------------------------------------
# 11. Generate answer
# --------------------------------------------------

answer = rag_chain.invoke(question)

print("\n========== FINAL ANSWER ==========\n")
print(answer)

C:\Users\admin\.conda\envs\insurance_llm\Lib\site-packages\langchain_google_genai\chat_models.py:4059: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



========== FINAL ANSWER ==========

Based on the provided document, an employee must complete at least 90 calendar days of employment before requesting a regular remote-work arrangement.
